In [40]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- 1. Importation des fichiers ---
data1 = pd.read_csv("../data/Données SPY.csv")
data2 = pd.read_csv("../data/AAPL.csv")
data3 = pd.read_csv("../data/USD_CNY.csv")
data4 = pd.read_csv("../data/DTB6.csv")
data5 = pd.read_csv("../data/HIS.csv")
data6 = pd.read_csv("../data/Gold.csv")
data7 = pd.read_csv("../data/TCD6.csv")
data8 = pd.read_csv("../data/DR5.xls")
data9 = pd.read_csv("../data/Taux_ST6.xls")
data10 = pd.read_csv("../data/VRV.xls")

# --- 2. Uniformisation des colonnes et conversion de la date ---
for df in [data1, data2, data3, data4, data5, data6, data7, data8, data9, data10]:
    df.columns = [col.strip().capitalize() for col in df.columns]
    if 'Unnamed: 0' in df.columns:
        df.rename(columns={'Unnamed: 0':'Date'}, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# --- 3. Calcul des rendements avant fusion (corrigé) ---
# Nettoyage spécifique pour le fichier HIS
data5['Price'] = data5['Price'].astype(str)
data5['Price'] = data5['Price'].str.replace(',', '', regex=False)  # supprimer les virgules
data5['Price'] = pd.to_numeric(data5['Price'], errors='coerce')

col_names = ['SPY','AAPL','DollarYuan','DTB6','HIS','Or','DCD6M', 'DR5', 'Taux_ST6', 'VRV']
col_prices = ['Price','Price','Price','Dtb6','Price','Price','Dcd6m','Price', 'Price', 'Price']
dataframes = [data1, data2, data3, data4, data5, data6, data7, data8, data9, data10]

for df, name, price in zip(dataframes, col_names, col_prices):
    if price in df.columns:
        # Convertir la colonne en numérique avant le calcul
        df[price] = pd.to_numeric(df[price], errors='coerce')
        # Calcul du rendement journalier (en proportion, pas en %)
        df[name] = df[price].pct_change()
        # Supprimer les premières lignes NaN
        df.dropna(subset=[name], inplace=True)

# --- 4. Fusion des fichiers sur la colonne Date avec outer join ---
# On part du SPY (déjà converti en rendement)
data_merged = data1[['Date','SPY']]

# Liste des fichiers restants à fusionner
dfs = [data2, data3, data4, data5, data6, data7, data8, data9, data10]
col_names = ['AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV' ]
col_prices = ['AAPL','DollarYuan','DTB6','Price','Or','DCD6M','Dr5', 'St6', 'Vrv']  #  on fusionne sur les rendements déjà calculés

for df, col_name, price_col in zip(dfs, col_names, col_prices):
    data_merged = data_merged.merge(df[['Date', price_col]], on='Date', how='outer')
    data_merged.rename(columns={price_col: col_name}, inplace=True)

print("Nombre de lignes après fusion :", len(data_merged))

# --- 5. Remplissage des valeurs manquantes ---
data_merged.fillna(method='ffill', inplace=True)
data_merged.fillna(method='bfill', inplace=True)
for col in ['SPY','AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV' ]:
    data_merged[col] = pd.to_numeric(data_merged[col], errors='coerce')
    if data_merged[col].isna().sum() > 0:
        data_merged[col].fillna(data_merged[col].mean(), inplace=True)

# --- 6. Supprimer les colonnes trop vides (>50% NaN) ---
X_multi = data_merged[['AAPL','DollarYuan','DTB6','HIS','Or','DCD6M','DR5', 'Taux_ST6', 'VRV']]
X_multi = X_multi.loc[:, X_multi.isna().mean() < 0.5]
y = data_merged[['SPY']]

# Nettoyage final : supprimer les lignes restantes avec NaN
data_clean = pd.concat([X_multi, y], axis=1).dropna()
X_multi = data_clean.drop(columns=['SPY'])
y = data_clean[['SPY']]

print("Lignes finales après nettoyage :", len(data_clean))
print("Colonnes finales :", X_multi.columns.tolist())

# --- 7. Régression multiple ---
X_multi_sm = sm.add_constant(X_multi)
mon_modele = sm.OLS(y, X_multi_sm).fit()
print(mon_modele.summary())

Nombre de lignes après fusion : 1305
Lignes finales après nettoyage : 1305
Colonnes finales : ['AAPL', 'DollarYuan', 'DTB6', 'HIS', 'Or', 'DCD6M', 'DR5', 'Taux_ST6', 'VRV']
                            OLS Regression Results                            
Dep. Variable:                    SPY   R-squared:                       0.241
Model:                            OLS   Adj. R-squared:                  0.236
Method:                 Least Squares   F-statistic:                     45.75
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           9.74e-72
Time:                        18:16:37   Log-Likelihood:                 3805.2
No. Observations:                1305   AIC:                            -7590.
Df Residuals:                    1295   BIC:                            -7539.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err     